# Data Exploration

**Quito Public Transport Intelligence Platform** — notebook 01 of the pipeline.

Establishes what the three raw datasets contain, which time window is trustworthy,
and which measures can be used downstream.

Every section follows the same shape: a line on *why* the check is run, the code, then a
**Finding** block recording the result. The findings are the deliverable — this notebook
is the data-quality record for the project, not a scratchpad.

---

## 0. Configuration

Two windows, deliberately different. `RAW_*` is everything that was collected;
`LOCAL_*` is the subset that survives the quality checks in sections 5–7.

In [0]:
import os
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/data-engineering/quito-transport-platform/s3_raw"

# Full span present in the raw files
RAW_START, RAW_END = "2026-08-20", "2026-09-02"

# Analysis window: complete LOCAL days with hourly traffic coverage (derived in section 6)
LOCAL_START, LOCAL_END = "2026-08-25", "2026-08-31"

UTC_OFFSET_HOURS = 5   # Quito is UTC-5 year-round; Ecuador has no DST

---

## 1. Physical layout

Walk the tree before reading anything. Partition keys, nesting depth and file counts are all
visible from the filesystem and cost nothing to inspect.

In [0]:
for root, dirs, files in os.walk(BASE):
    depth = root[len(BASE):].count(os.sep)
    if depth <= 3:
        print(f"{'  ' * depth}{os.path.basename(root)}/  ({len(files)} files)")

s3_raw/  (0 files)
  network/  (0 files)
    year=2026/  (0 files)
      month=08/  (0 files)
      month=09/  (0 files)
      month=10/  (0 files)
  traffic/  (0 files)
    year=2026/  (0 files)
      month=08/  (0 files)
      month=09/  (0 files)
  weather/  (0 files)
    year=2026/  (0 files)
      month=06/  (0 files)
      month=08/  (0 files)
      month=09/  (0 files)
      month=10/  (0 files)


`traffic` reports 0 files at the day level because it has a fourth partition level the others lack:

In [0]:
print(sorted(os.listdir(f"{BASE}/traffic/year=2026/month=08/day=20")))

['hour=00', 'hour=01', 'hour=06', 'hour=12', 'hour=18']


> **Finding**
>
> This is not one partitioned dataset but three independent ones.
>
> - `network` and `weather` are partitioned `year / month / day`
> - `traffic` adds `hour`
>
> Pointing Spark at `BASE` raises `CONFLICTING_DIRECTORY_STRUCTURES`. Each dataset must be read separately.

---

## 2. Schemas

Parquet keeps its schema in the file footer, so this reads one footer per dataset — no data scanned.

In [0]:
traffic_raw = spark.read.parquet(f"{BASE}/traffic")
weather_raw = spark.read.parquet(f"{BASE}/weather")
network_raw = spark.read.parquet(f"{BASE}/network")

for name, df in [("traffic", traffic_raw), ("weather", weather_raw), ("network", network_raw)]:
    print(f"=== {name} ===")
    df.printSchema()

=== traffic ===
root
 |-- route_id: string (nullable = true)
 |-- route_ref: string (nullable = true)
 |-- route_name: string (nullable = true)
 |-- sampled_at: string (nullable = true)
 |-- local_hour: long (nullable = true)
 |-- local_weekday: long (nullable = true)
 |-- waypoint_count: long (nullable = true)
 |-- mean_hop_km: double (nullable = true)
 |-- distance_meters: double (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- mean_speed_kmh: double (nullable = true)
 |-- segments_total: long (nullable = true)
 |-- segments_known: long (nullable = true)
 |-- congestion_known_pct: double (nullable = true)
 |-- congestion_low: long (nullable = true)
 |-- congestion_moderate: long (nullable = true)
 |-- congestion_heavy: long (nullable = true)
 |-- congestion_severe: long (nullable = true)
 |-- congestion_unknown: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer

> **Finding** — three different grains, not three peer tables
>
> | dataset | grain | role |
> |---|---|---|
> | `traffic` | route x hour | fact |
> | `weather` | time only, no location column | time dimension |
> | `network` | route x stop, re-fetched daily | slowly-changing dimension |
>
> Two consequences for the model:
>
> - Weather is a **single observation point for the whole city**, so the join is time-only.
>   Nothing can be said about weather varying by route.
> - `network` is a **daily snapshot**. Joining it on date would fan the fact table out by the number of days.

---

## 3. Raw volumes

Filtering on the partition columns lets Spark prune directories instead of scanning everything.

In [0]:
def scan_window(df):
    return df.filter(F.make_date("year", "month", "day").between(RAW_START, RAW_END))

traffic = scan_window(traffic_raw)
weather = scan_window(weather_raw)
network = scan_window(network_raw)

print("traffic", traffic.count())
print("weather", weather.count())
print("network", network.count())

traffic 3108
weather 574
network 27435


> **Finding**
>
> | dataset | rows |
> |---|---|
> | `traffic` | 3 108 |
> | `weather` | 574 |
> | `network` | 27 435 |
>
> 574 / 14 days = **exactly 41 rows per day**. That is not hourly observation data.
> 41 = 1 + 40, and 40 is what OpenWeather's 5-day / 3-hour forecast endpoint returns.
> Tested in section 4.

---

## 4. Weather composition

`record_type` exists for a reason, and the usual reason is that observations and forecasts share the file.

In [0]:
weather.groupBy("record_type") \
       .count() \
       .display()

record_type,count
current,14
forecast,560


`timestamp` and `fetched_at` arrive as strings, so they sort as text rather than as time. Parse both:

- `ts` — when the weather happened
- `fetched` — when the API was called

A large gap between the two is the signature of a forecast.

In [0]:
w = weather \
    .withColumn("ts", F.to_timestamp("timestamp")) \
    .withColumn("fetched", F.to_timestamp("fetched_at"))

w.filter(F.make_date("year", "month", "day") == F.lit("2026-08-20")) \
 .select("record_type", "fetched", "ts", "temp_c") \
 .orderBy("ts") \
 .display()

record_type,fetched,ts,temp_c
current,2026-08-20T05:00:04.982Z,2026-08-20T05:00:05.000Z,10.33
forecast,2026-08-20T05:00:04.982Z,2026-08-20T06:00:00.000Z,10.33
forecast,2026-08-20T05:00:04.982Z,2026-08-20T09:00:00.000Z,10.29
forecast,2026-08-20T05:00:04.982Z,2026-08-20T12:00:00.000Z,10.21
forecast,2026-08-20T05:00:04.982Z,2026-08-20T15:00:00.000Z,17.07
forecast,2026-08-20T05:00:04.982Z,2026-08-20T18:00:00.000Z,17.77
forecast,2026-08-20T05:00:04.982Z,2026-08-20T21:00:00.000Z,15.53
forecast,2026-08-20T05:00:04.982Z,2026-08-21T00:00:00.000Z,11.88
forecast,2026-08-20T05:00:04.982Z,2026-08-21T03:00:00.000Z,9.49
forecast,2026-08-20T05:00:04.982Z,2026-08-21T06:00:00.000Z,7.83


> **Finding**
>
> 14 `current` + 560 `forecast`, and 560 = 14 x 40. One API call per day at 05:00 UTC (midnight local)
> wrote one observation plus a 5-day forecast at 3-hour steps.
>
> **Timestamps are UTC.** The `+00:00` suffix says so, and the temperature curve agrees —
> 10.3 C at 05:00 UTC and 17.8 C at 18:00 UTC are midnight and 1 pm local, the right shape for Quito.
>
> **Only 14 real observations exist**, one per day, all at midnight local. Not enough for an hourly join.
>
> The forecasts rescue this. Each target time is forecast several times at different lead times, so
> keeping the shortest-lead forecast per target yields a usable 3-hourly series — with lead time
> retained as a column so the provenance stays honest.

---

## 5. Traffic time base

Three time representations exist in `traffic`: the `hour` partition, the `local_hour` column, and the
`sampled_at` string. Weather is UTC; if `hour` turned out to be local, every joined reading would be
five hours out of place.

In [0]:
t = traffic \
    .withColumn("ts", F.to_timestamp("sampled_at"))

t.groupBy("hour", "local_hour") \
 .count() \
 .orderBy("hour") \
 .display()

hour,local_hour,count
0,19,196
1,20,140
2,21,112
3,22,112
4,23,112
5,0,112
6,1,182
7,2,112
8,3,112
9,4,112


> **Finding**
>
> `local_hour = hour - 5` for every row, with no hour splitting across two local hours.
> So **`hour` is UTC** and `local_hour` is Quito time. No DST ambiguity.
>
> This matters for the join: the weather forecast grid sits at 00, 03, 06 … 21 UTC and traffic's
> partition is UTC too, so those hours align exactly with no conversion.

---

## 6. Coverage grid

`pivot` turns each distinct hour into its own column, producing a day-by-hour heatmap of row counts.
Nulls are hours never sampled; uneven numbers within a row mean the route set changed mid-day.

In [0]:
t.withColumn("date", F.make_date("year", "month", "day")) \
 .groupBy("date") \
 .pivot("hour") \
 .count() \
 .orderBy("date") \
 .display()

date,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
2026-08-20,14,14,null,null,null,null,14,null,null,null,null,null,14,null,null,null,null,null,14,null,null,null,null,null
2026-08-21,14,null,null,null,null,null,14,null,null,null,null,null,14,null,null,null,null,null,14,null,null,null,null,null
2026-08-22,14,null,null,null,null,null,14,null,null,null,null,null,14,null,null,null,null,null,14,null,null,null,null,null
2026-08-23,14,null,null,null,null,null,14,null,null,null,null,null,14,null,null,null,null,null,14,null,null,null,null,null
2026-08-24,14,null,null,null,null,null,14,null,null,null,null,null,14,null,null,null,14,14,14,14,14,14,14,14
2026-08-25,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14
2026-08-26,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14
2026-08-27,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14
2026-08-28,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14
2026-08-29,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14


> **Finding**
>
> Every non-null cell is exactly **14** — fourteen routes, no exceptions. But the collection cadence
> changed mid-run:
>
> | period (UTC) | cadence |
> |---|---|
> | Aug 20 – Aug 24 12:00 | 4 samples/day, hours 0 / 6 / 12 / 18 |
> | Aug 24 16:00 – Sep 01 23:00 | hourly, complete |
> | Sep 02 | truncated at 01:00 |
>
> The arithmetic reconciles: 70 + 168 + 154 + 2688 + 28 = 3 108, the exact row count.
> Nothing missing, nothing duplicated.
>
> Mixing the sparse regime with the hourly one would bias every hourly average toward hours 0/6/12/18,
> so the sparse days are dropped. The research questions are local-time questions — rush hour,
> weekday effects — so the window is cut to complete **local** days:
>
> **2026-08-25 to 2026-08-31. Seven days, exactly one calendar week.**

---

## 7. Lock the analysis window

Build a real timestamp from the partition columns, shift it to local time, filter on the local date.

Deriving the window rather than hardcoding date conditions has a second payoff: it produces
`local_date`, which every daily aggregation downstream reuses.

In [0]:
utc_ts = F.expr("make_timestamp(year, month, day, hour, 0, 0)")

analysis = t \
    .withColumn("utc_ts", utc_ts) \
    .withColumn("local_ts", utc_ts - F.expr(f"INTERVAL {UTC_OFFSET_HOURS} HOURS")) \
    .withColumn("local_date", F.to_date("local_ts")) \
    .filter(F.col("local_date").between(LOCAL_START, LOCAL_END))

print("rows:", analysis.count(), "(expected 2352 = 7 days x 24 h x 14 routes)")

rows: 2352 (expected 2352 = 7 days x 24 h x 14 routes)


Cross-check the derived local time against the `local_hour` column the ingestion pipeline wrote.
Two independent derivations agreeing is worth more than either alone.

In [0]:
print("timezone mismatches:", analysis.filter(F.hour("local_ts") != F.col("local_hour")).count())

timezone mismatches: 0


In [0]:
analysis.groupBy("route_id", "route_ref", "route_name") \
        .count() \
        .orderBy("route_ref") \
        .display()

route_id,route_ref,route_name,count
12113848,C1,Trolebus C1 El Recreo => El Labrador,168
12113850,C1,Trolebus C1 El Labrador => El Recreo,168
85965,C4,Trolebus C4 Quitumbe => La Colón,168
2006760,C4,Trolebus C4 La Colón => Quitumbe,168
12113852,C6,Trolebus C6 El Recreo => Quitumbe,168
85972,C6,Trolebus C6 Quitumbe - El Recreo,168
12110365,E1,Ecovía E1 T. Sur - Las Universidades,168
12110382,E1R,Ecovía E1R T. Sur - Terminal El Recreo,168
12113855,E4,Ecovia E4 Playón de la Marín - Quitumbe,168
12113856,E4,Ecovia E4 Quitumbe - Playón de la Marín,168


> **Finding**
>
> 2 352 rows · 0 timezone mismatches · 14 routes at 168 samples each (7 x 24).
>
> The fact table is a perfect rectangle. No route dropped out mid-week.

---

## 8. Nulls and ranges

`isNull()` returns true/false per row; casting to int and summing turns that into a null count per
column. Which columns fail *together* usually reveals why.

In [0]:
MEASURES = ["waypoint_count", "mean_hop_km", "distance_meters", "duration_seconds",
            "mean_speed_kmh", "segments_total", "segments_known", "congestion_known_pct",
            "congestion_low", "congestion_moderate", "congestion_heavy",
            "congestion_severe", "congestion_unknown"]

analysis.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in MEASURES]) \
        .display()

waypoint_count,mean_hop_km,distance_meters,duration_seconds,mean_speed_kmh,segments_total,segments_known,congestion_known_pct,congestion_low,congestion_moderate,congestion_heavy,congestion_severe,congestion_unknown
0,0,0,0,0,0,0,0,0,0,0,0,0


Percentiles expose the impossible values routing APIs commonly emit — zero or 300 km/h speeds,
zero-second durations, negative distances, percentages above 100 — and show whether a long tail
would distort any mean.

In [0]:
analysis.select("mean_speed_kmh", "duration_seconds", "distance_meters", "congestion_known_pct") \
        .summary("min", "25%", "50%", "75%", "max") \
        .display()

summary,mean_speed_kmh,duration_seconds,distance_meters,congestion_known_pct
min,18.03,855.2,8460.8,0.0
25%,21.49,1955.2,13804.2,12.1
50%,23.49,2778.9,19057.6,37.4
75%,30.55,3506.7,25796.0,56.4
max,39.74,5341.0,31036.9,95.3


> **Finding**
>
> Zero nulls across all thirteen measures.
>
> | measure | min | median | max | verdict |
> |---|---|---|---|---|
> | `mean_speed_kmh` | 18.0 | 23.5 | 39.7 | plausible |
> | `duration_seconds` | 855 | 2 779 | 5 341 | 14–89 min, plausible |
> | `distance_meters` | 8 461 | 19 058 | 31 037 | 8.5–31 km, plausible |
> | `congestion_known_pct` | 0.0 | 37.4 | 95.3 | **weak** |
>
> `mean_speed_kmh` derives from distance and duration, which the routing API returns regardless of
> traffic-layer coverage. Speed is trustworthy.
>
> `congestion_known_pct` is not. In half the samples fewer than 37 % of road segments reported
> congestion data; the bottom quartile is under 12 %; some samples have none at all. Comparing
> `congestion_heavy` between a row at 80 % coverage and one at 10 % compares different things.

---

## 9. Is the missingness structured?

Random gaps are a precision problem. Gaps that correlate with route or time of day are a **bias**
problem. This decides whether congestion is usable at all.

In [0]:
analysis.groupBy("route_id", "route_ref") \
        .agg(F.round(F.min("distance_meters"), 0).alias("min_dist"), \
             F.round(F.max("distance_meters"), 0).alias("max_dist"), \
             F.round(100 * (F.max("distance_meters") - F.min("distance_meters")) \
                     / F.min("distance_meters"), 1).alias("spread_pct"), \
             F.round(F.avg("congestion_known_pct"), 1).alias("avg_known")) \
        .orderBy(F.desc("spread_pct")) \
        .display()

route_id,route_ref,min_dist,max_dist,spread_pct,avg_known
85972,C6,17391.0,20901.0,20.2,28.8
12113854,E6,10375.0,11869.0,14.4,25.0
12113852,C6,21419.0,24301.0,13.5,31.7
85965,C4,26420.0,29552.0,11.9,39.2
2006760,C4,28021.0,31037.0,10.8,33.9
12113848,C1,20382.0,21983.0,7.9,47.3
2021069,Metro-Bus Corredor Central Norte,26142.0,28218.0,7.9,43.0
12113850,C1,18984.0,20091.0,5.8,49.4
85969,Metrobus-Q,25506.0,26928.0,5.6,41.7
12113856,E4,13359.0,14090.0,5.5,32.3


Two questions in one table.

- **Coverage** — if poor coverage concentrates in a few routes, drop those routes rather than the
  whole column.
- **Geometry stability** — a bus route's path is fixed, so min and max distance should be nearly
  identical. A spread beyond a percent or two means the routing API returned different paths on
  different calls, and speed is not comparable across time for that route.

In [0]:
analysis.groupBy("local_hour") \
        .agg(F.round(F.avg("congestion_known_pct"), 1).alias("avg_known"),
             F.round(F.avg("mean_speed_kmh"), 1).alias("avg_speed")) \
        .orderBy("local_hour") \
        .display()

local_hour,avg_known,avg_speed
0,20.6,33.5
1,11.2,34.0
2,6.8,34.2
3,7.9,33.8
4,1.8,33.1
5,4.2,28.8
6,9.4,23.4
7,23.6,21.4
8,31.9,21.4
9,38.1,22.6


The concern: if congestion data is reported mainly during busy daytime hours and thins out overnight,
the missingness correlates with exactly the variable under study, and any naive hourly congestion
average is biased.

`avg_speed` sits alongside so we can see whether speed tells a coherent rush-hour story on its own.
If it does, speed becomes the primary metric and congestion drops to supporting detail.

## Step 10 — Does the path change with the hour?

---

## Summary

### Established

- Three independent datasets at three grains; read and modelled separately.
- All timestamps are UTC. Quito is UTC-5, no DST. `traffic.local_hour` verified correct.
- Analysis window: **2026-08-25 to 2026-08-31 local** — seven days, one full calendar week.
- Fact table: **2 352 rows = 14 routes x 24 hours x 7 days.** Complete, no nulls, no duplicates.
- `mean_speed_kmh`, `duration_seconds` and `distance_meters` are trustworthy.

### Known limitations

Carry these into the README rather than leaving them to be discovered.

- Weather has no spatial dimension — one observation point for the whole city.
- Only 14 weather observations exist; the hourly series is reconstructed from shortest-lead forecasts.
- `congestion_*` describes a varying minority of segments, median 37 %.
- 2 352 rows is small. Enough for descriptive analysis and a clean pipeline; not enough for modelling.

### Open questions

- Is congestion missingness structured by route or by hour? *(section 9)*
- Is route geometry stable over time? *(section 9)*
- Does `network` change across daily snapshots, or is one snapshot enough?
- Do `traffic.route_id` and `network.route_id` fully overlap?